# Bulk RNA-seq 파이프라인 — Colab

전체 Snakemake 파이프라인(DESeq2 · GSEA · ORA · TFEA · PROGENy · cMap → HTML 리포트)을 실행.

**필요한 것:**
nf-core/rnaseq 분석 결과
- `salmon.merged.gene_counts_length_scaled.tsv`
- `multiqc_report_data/` 디렉터리를 **zip으로 압축한 파일** (Colab은 폴더 업로드 불가)

**결과물:** `report/report.html`과 `results.zip`

**소요 시간:** 첫 실행 약 10분, 같은 세션에서 재실행 시 1–2분.

## 1. conda + Snakemake 설치

이 세션에서 한 번만 실행.

> **첫 실행 안내**: 이 셀이 Colab 시스템 lib(libstdc++ / libssl / libcrypto) 일부를 miniforge 빌드로 교체하고 **커널을 한 번 재시작**할 수 있음. 재시작 후 같은 셀을 다시 실행하면 swap이 no-op이 되어 그대로 설치 진행. warm runtime이거나 Colab 이미지가 이미 최신이면 재시작 없음.

In [ ]:
%%time
import os, subprocess, glob, shutil

# --- Miniforge (idempotent) -------------------------------------------------
if not os.path.exists('/content/miniforge/bin/conda'):
    !wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/miniforge.sh
    !bash /tmp/miniforge.sh -b -p /content/miniforge
    !rm /tmp/miniforge.sh
os.environ['PATH'] = '/content/miniforge/bin:' + os.environ['PATH']

# --- 시스템 lib ABI patch (Colab 베이스 2026-04 기준 conda-forge보다 뒤처짐) -----
# WHY: libstdc++에 CXXABI_1.3.15 없음(icu78), libssl/libcrypto에 OPENSSL_3.2.0
# 없음(curl4). 커널 시작 시 이미 로드된 lib이라 in-process preload로는 SONAME을
# 못 이김 — 시스템 파일을 덮어쓰고 커널을 강제 재시작. 심볼이 이미 있는 경우
# (warm runtime / 향후 Colab 이미지 업데이트)엔 swap 자체를 스킵.
# 제거: 새 런타임에서 이 셀 실행 전에 두 `strings ... | grep -c <symbol>`이 모두
# >0이면 통째로 삭제 가능. ABI는 하위호환이라 안전.
def _has_symbol(path, sym):
    try:
        return sym in subprocess.check_output(['strings', path], stderr=subprocess.DEVNULL).decode('latin-1', 'ignore')
    except Exception:
        return False

NEEDED = [('libstdc++.so.6', 'CXXABI_1.3.15'),
          ('libssl.so.3',    'OPENSSL_3.2.0'),
          ('libcrypto.so.3', 'OPENSSL_3.2.0')]

def _swap_lib(libname):
    paths = [f'/content/miniforge/lib/{libname}'] + glob.glob(f'/content/miniforge/lib/{libname}.*')
    cands = sorted({os.path.realpath(p) for p in paths if os.path.exists(p)} - {''})
    cands = [p for p in cands if os.path.isfile(p)]
    if not cands:
        return False
    dst = f'/usr/lib/x86_64-linux-gnu/{libname}'
    if os.path.exists(dst + '.bak'):
        return False
    shutil.move(dst, dst + '.bak')
    shutil.copy2(cands[-1], dst)
    print(f'  swapped {libname} <- {cands[-1]}')
    return True

todo = [name for name, sym in NEEDED if not _has_symbol(f'/usr/lib/x86_64-linux-gnu/{name}', sym)]
if todo:
    print(f'Swap 대상 시스템 lib: {todo}')
    if any(_swap_lib(n) for n in todo):
        print('커널 재시작 — 런타임 복귀 후 이 셀을 다시 실행하세요.')
        os.kill(os.getpid(), 9)
else:
    print('시스템 lib 정상, swap 불필요.')

# --- base env에 Snakemake 설치 ---------------------------------------------
!mamba install -y -n base -c conda-forge -c bioconda snakemake pandas pyyaml 2>&1 | tail -5
!snakemake --version


## 2. 파이프라인 코드 가져오기

In [ ]:
!rm -rf /content/bulk-rnaseq
!git clone --depth=1 https://github.com/artblakey19/BulkRNAseq-Analyzer.git /content/bulk-rnaseq
%cd /content/bulk-rnaseq


## 3. 데이터 업로드

1. **Counts TSV** — `salmon.merged.gene_counts_length_scaled.tsv`
2. **MultiQC 데이터** — `multiqc_report_data/`를 하나의 `.zip`으로 압축 (Colab은 폴더를 직접 받을 수 없음)

In [ ]:
# Counts TSV 업로드
from google.colab import files
uploaded = files.upload()
counts_file = next(iter(uploaded))
!mv "{counts_file}" /content/bulk-rnaseq/counts.tsv
!ls -la counts.tsv
print('헤더:')
!head -1 counts.tsv | tr '\t' '\n'

In [ ]:
# multiqc_report_data.zip 업로드 후 해제
from google.colab import files
uploaded = files.upload()
zip_file = next(iter(uploaded))
!rm -rf multiqc_report_data
!unzip -q "{zip_file}" -d /tmp/mqc_unpack
# "multiqc_report_data.zip 안에 multiqc_report_data/ 폴더가 있는 경우"와 "zip에 파일이 바로 있는 경우" 모두 처리
!if [ -d /tmp/mqc_unpack/multiqc_report_data ]; then mv /tmp/mqc_unpack/multiqc_report_data /content/bulk-rnaseq/; else mv /tmp/mqc_unpack /content/bulk-rnaseq/multiqc_report_data; fi
!rm -f "{zip_file}"
!ls multiqc_report_data | head -10

## 4. 프로젝트 메타데이터

메타데이터 입력, HTML 보고서에 들어가는 내용.

In [ ]:
#@markdown ### 프로젝트
project_name = "My RNA-seq study"  #@param {type:"string"}
analyst = ""  #@param {type:"string"}

#@markdown ### 실험 정보 (전부 선택 입력)
cell_line = ""  #@param {type:"string"}
compound = ""  #@param {type:"string"}
dose = ""  #@param {type:"string"}
duration = ""  #@param {type:"string"}
notes = ""  #@param {type:"string"}

print('프로젝트 메타데이터 저장 완료.')

## 5. 샘플·비교군 정의

분석에 필요한 두 가지 정보를 지정.

- **SAMPLES** — 각 샘플이 어느 **그룹(condition)** 에 속하는지. 예컨대 샘플 4개 중 앞 2개가 대조군·뒤 2개가 처리군이면 `{"Ctrl_1": "Control", "Ctrl_2": "Control", "Tx_1": "Treatment", "Tx_2": "Treatment"}`. 그룹 이름은 자유 (DMSO·Drug, WT·KO 등).
- **CONTRASTS** — 어느 그룹들을 서로 비교할지. `(관심군, 대조군)` 쌍으로 지정. Control이 뒤에 와야 FoldChange > 0이 관심군에서 대조군 대비 발현량이 증가했다는 의미가 됨.

### 작업 순서

1. **아래 첫 번째 코드 셀을 실행** — counts 파일 헤더에서 샘플 이름을 읽어 SAMPLES 템플릿을 출력함.
2. **두 번째 코드 셀 편집** — 출력된 템플릿을 그대로 붙여넣고 `"CHANGE_ME"` 자리에 각 샘플의 그룹 이름을 채움. 이어서 `CONTRASTS` 리스트도 원하는 조합으로 수정.

In [ ]:
# counts 헤더에서 샘플 감지 (3번째 열부터가 샘플)
from pathlib import Path

counts_path = Path('/content/bulk-rnaseq/counts.tsv')
if not counts_path.exists():
    # 업로드를 건너뛴 경우 test fixture로 fallback
    counts_path = Path('/content/bulk-rnaseq/tests/test_data/counts.tsv')
    print('(test fixture 사용 — 사용자 데이터 미업로드)')

header = counts_path.read_text().splitlines()[0].split('\t')
detected_samples = header[2:]
print(f'샘플 {len(detected_samples)}개 감지:\n')
print('SAMPLES = {')
for s in detected_samples:
    print(f'    {s!r}: "CHANGE_ME",')
print('}')

In [ ]:
### ← EDIT ### — 위 셀 출력을 여기 붙여넣고 각 condition 값을 설정.
SAMPLES = {
    "Control_1":   "Control",
    "Control_2":   "Control",
    "Treatment_1": "Treatment",
    "Treatment_2": "Treatment",
}

### ← EDIT ### — 원하는 비교를 모두 나열 (numerator vs denominator).
# numerator = 관심 condition (예: treated)
# denominator = 기준 condition (예: control)
CONTRASTS = [
    # (contrast_id, numerator, denominator, description)
    ("Treatment_vs_Control", "Treatment", "Control", "Treatment vs Control"),
]

## 6. config 파일 생성

위 폼 입력과 SAMPLES/CONTRASTS로부터 `config/config.yaml`, `config/samples.tsv`, `config/contrasts.tsv`를 생성. 분석 설정(DE cutoff, MSigDB collection 등)은 `config.template.yaml`내용을 기준으로 사용.

In [ ]:
import csv
from collections import Counter
from pathlib import Path
import yaml

repo = Path('/content/bulk-rnaseq')
config_path = repo / 'config' / 'config.yaml'
samples_path = repo / 'config' / 'samples.tsv'
contrasts_path = repo / 'config' / 'contrasts.tsv'
template_path = repo / 'config' / 'config.template.yaml'

# 사용할 counts/multiqc 경로 결정
user_counts = repo / 'counts.tsv'
user_mqc = repo / 'multiqc_report_data'
if user_counts.exists() and user_mqc.exists():
    counts_rel = 'counts.tsv'
    mqc_rel = 'multiqc_report_data'
    print('업로드 데이터 사용.')
else:
    counts_rel = 'tests/test_data/counts.tsv'
    mqc_rel = 'tests/test_data/multiqc_report_data'
    print('번들된 test fixture 사용.')

cfg = yaml.safe_load(template_path.read_text())
cfg['project']['name'] = project_name
cfg['project']['analyst'] = analyst
cfg['project']['experiment'] = {
    'cell_line': cell_line, 'compound': compound, 'dose': dose,
    'duration': duration, 'notes': notes,
}
cfg['input']['counts_tsv'] = counts_rel
cfg['input']['multiqc_data_dir'] = mqc_rel
cfg['input']['samples_tsv'] = 'config/samples.tsv'
cfg['input']['contrasts_tsv'] = 'config/contrasts.tsv'

# SAMPLES를 counts 헤더와 대조 검증
header = (repo / counts_rel).read_text().splitlines()[0].split('\t')[2:]
missing = [s for s in header if s not in SAMPLES]
extra = [s for s in SAMPLES if s not in header]
if missing or extra:
    raise SystemExit(
        f'SAMPLES dict 불일치.\n'
        f'  counts엔 있으나 SAMPLES에 없음: {missing}\n'
        f'  SAMPLES에만 있고 counts엔 없음: {extra}'
    )
unknown = [c for c in SAMPLES.values() if c == 'CHANGE_ME' or not c]
if unknown:
    raise SystemExit('SAMPLES 전 항목의 값을 채우세요 (CHANGE_ME/빈값 불가)')

# samples.tsv — condition별로 replicate 번호 자동 부여
replicate_of = Counter()
sample_rows = []
for sid in header:  # 열 순서 보존
    cond = SAMPLES[sid]
    replicate_of[cond] += 1
    sample_rows.append({'sample': sid, 'condition': cond,
                         'replicate': replicate_of[cond], 'batch': 1})

# contrasts.tsv
conds = set(SAMPLES.values())
contrast_rows = []
for cid, num, den, desc in CONTRASTS:
    if num not in conds or den not in conds:
        raise SystemExit(f'contrast {cid}: {num!r} 또는 {den!r}가 conditions {conds}에 없음')
    contrast_rows.append({'contrast_id': cid, 'factor': 'condition',
                           'numerator': num, 'denominator': den, 'description': desc})

config_path.parent.mkdir(parents=True, exist_ok=True)
with config_path.open('w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, default_flow_style=False)

def write_tsv(p, rows, cols):
    with p.open('w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=cols, delimiter='\t')
        w.writeheader(); w.writerows(rows)

write_tsv(samples_path, sample_rows, ['sample', 'condition', 'replicate', 'batch'])
write_tsv(contrasts_path, contrast_rows,
          ['contrast_id', 'factor', 'numerator', 'denominator', 'description'])

print('생성:')
print(f'  {config_path.relative_to(repo)}')
print(f'  {samples_path.relative_to(repo)} ({len(sample_rows)} 행)')
print(f'  {contrasts_path.relative_to(repo)} ({len(contrast_rows)} 행)')

## 7. Dry-run (실제 실행 없이 DAG만 검증, Optional)

In [ ]:
!snakemake \
  --snakefile workflow/Snakefile \
  --configfile config/config.yaml \
  --dry-run 2>&1 | tail -30

## 8. 파이프라인 실행

첫 실행 약 10분 (conda env 설치). 같은 Colab 세션에서 재실행 시 1–2분, `--keep-going` flag는 실패한 분석(예: L2S2 네트워크 오류)이 있어도 나머지 작업을 계속 실행함.

In [ ]:
%%time
#@markdown ### 실행 옵션
cores = 2  #@param {type:"slider", min:1, max:8, step:1}
keep_going = True  #@param {type:"boolean"}

_flag = "--keep-going" if keep_going else ""
!snakemake --snakefile workflow/Snakefile --configfile config/config.yaml --use-conda --cores {cores} {_flag} 2>&1 | tail -80

## 9. 결과 다운로드

`results/` 아래 전체를 하나의 zip으로 묶음. HTML 리포트는 zip 내부 `report/report.html`.

In [ ]:
from pathlib import Path
from google.colab import files

report = Path('results/report/report.html')
if not report.exists():
    raise SystemExit('report.html이 없음 — 섹션 8 출력 또는 섹션 10 로그 확인')

!zip -qr results.zip results/
print(f'report 크기: {report.stat().st_size/1024:.1f} KB')
files.download('results.zip')

## 10. 문제 해결

rule별 로그를 덤프

In [ ]:
!for f in $(find logs -name '*.log' 2>/dev/null); do echo "=== $f ==="; tail -30 "$f"; echo; done

## 11. Explore — plot 다시 그리기

파이프라인이 끝나면 `results/`에 플롯을 다시 그리는 데 필요한 모든 파일이 들어 있음. 이 섹션은 Python 커널 안에서 R을 호스팅(`rpy2`의 `%%R` magic)하고, 파이프라인이 `.snakemake/conda/` 아래에 빌드해 둔 r-deseq2 env를 그대로 재사용함.

아래 setup 셀 한 번에 Explore 섹션이 필요한 R 패키지를 모두 깔아둠 (tidyverse + GSEA enrichment plot용 fgsea/msigdbr + L2S2 재쿼리용 httr/jsonlite). 이후엔 cutoff, top-N, GSEA pathway 이름, L2S2 signature 크기 등을 자유롭게 조정 가능.

### Setup — rpy2 설치 · R env 로드

In [ ]:
import glob, os, subprocess

# 섹션 1의 lib swap이 적용됐는지 — 우리가 만든 마커가 아니라 rpy2/curl이
# dlopen할 실제 심볼을 검사.
def _has_symbol(path, sym):
    try:
        return sym in subprocess.check_output(['strings', path], stderr=subprocess.DEVNULL).decode('latin-1', 'ignore')
    except Exception:
        return False

_missing = [(p, s) for p, s in [('/usr/lib/x86_64-linux-gnu/libstdc++.so.6', 'CXXABI_1.3.15'),
                                 ('/usr/lib/x86_64-linux-gnu/libssl.so.3',    'OPENSSL_3.2.0'),
                                 ('/usr/lib/x86_64-linux-gnu/libcrypto.so.3', 'OPENSSL_3.2.0')]
            if not _has_symbol(p, s)]
assert not _missing, f'시스템 lib에 필요한 심볼이 없음: {_missing} — 섹션 1을 다시 실행하세요.'

# 파이프라인이 빌드한 conda env 중 SummarizedExperiment(r-deseq2)가 있는 것을
# rpy2 호스트로 사용. 섹션 1의 커널 재시작으로 PATH가 날아갔을 수 있으니 복원.
os.environ['PATH'] = '/content/miniforge/bin:' + os.environ['PATH']

def has_pkg(r_bin, pkg):
    return subprocess.call([r_bin, '-e', f'suppressMessages(library({pkg}))'],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL) == 0

r_bins = sorted(glob.glob('/content/bulk-rnaseq/.snakemake/conda/*/bin/R'))
R_BIN = next((r for r in r_bins if has_pkg(r, 'SummarizedExperiment')), None)
assert R_BIN, 'SummarizedExperiment 가 있는 conda env를 못 찾음 — 섹션 8을 먼저 실행.'
ENV_PREFIX = os.path.dirname(os.path.dirname(R_BIN))

# Explore 섹션이 필요한 모든 R 패키지를 mamba 한 번에 설치.
# tidyverse: 모든 plot 셀, fgsea/msigdbr: GSEA enrichment plot,
# httr/jsonlite: L2S2 재쿼리. 여기서 다 깔아두면 이후 셀이 "그냥 실행"됨.
WANT = {'r-yaml':'yaml', 'r-readr':'readr', 'r-dplyr':'dplyr', 'r-tidyr':'tidyr',
        'r-ggplot2':'ggplot2', 'r-scales':'scales', 'r-forcats':'forcats',
        'bioconductor-fgsea':'fgsea', 'r-msigdbr':'msigdbr',
        'r-httr':'httr', 'r-jsonlite':'jsonlite'}
need = [conda_pkg for conda_pkg, r_pkg in WANT.items() if not has_pkg(R_BIN, r_pkg)]
if need:
    print(f'Installing into {ENV_PREFIX}: {need}')
    subprocess.check_call(['mamba', 'install', '-y', '--prefix', ENV_PREFIX,
                           '-c', 'conda-forge', '-c', 'bioconda'] + need)
else:
    print('R 패키지 모두 준비됨.')

R_HOME = os.path.realpath(os.path.join(os.path.dirname(R_BIN), '..', 'lib', 'R'))
os.environ['R_HOME'] = R_HOME
os.environ['PATH']   = f'{os.path.dirname(R_BIN)}:' + os.environ['PATH']
print('R_HOME =', R_HOME)

!pip install --quiet rpy2
%load_ext rpy2.ipython


### R 설정 (한 번만)

In [ ]:
%%R
suppressPackageStartupMessages({
  library(yaml); library(readr); library(dplyr); library(tidyr)
  library(ggplot2); library(scales)
})
`%||%` <- function(a, b) if (is.null(a)) b else a

PROJECT_ROOT <- "/content/bulk-rnaseq"
rp <- function(...) file.path(PROJECT_ROOT, ...)
cfg       <- yaml::read_yaml(rp("config", "config.yaml"))
samples   <- read_tsv(rp(cfg$input$samples_tsv   %||% "config/samples.tsv"),   show_col_types = FALSE)
contrasts <- read_tsv(rp(cfg$input$contrasts_tsv %||% "config/contrasts.tsv"), show_col_types = FALSE)
CONTRAST <- contrasts$contrast_id[1]

safe_read <- function(path, sep = ",") {
  if (is.null(path) || !file.exists(path)) return(NULL)
  df <- tryCatch(
    if (sep == "\t") read_tsv(path, show_col_types = FALSE)
    else             read_csv(path, show_col_types = FALSE),
    error = function(e) NULL)
  if (is.null(df) || nrow(df) == 0) return(df)
  df
}
no_results <- function(msg) { message(msg); invisible(NULL) }
.trunc <- function(x, n = 50) ifelse(nchar(x) > n, paste0(substr(x, 1, n-1), "..."), x)

cat("Contrast:", CONTRAST, "\nAvailable:", paste(contrasts$contrast_id, collapse = ", "), "\n")


### QC

In [ ]:
%%R -w 700 -h 380
qc <- safe_read(rp("results", "qc", "qc_summary.tsv"), sep = "\t")
if (is.null(qc) || !"assigned_reads" %in% names(qc)) {
  no_results("QC summary not available.")
} else {
  df <- qc |> dplyr::mutate(sample = factor(sample, levels = sample),
                            condition = as.character(condition))
  ggplot(df, aes(sample, assigned_reads, fill = condition)) +
    geom_col() + scale_y_continuous(labels = scales::comma) +
    labs(x = NULL, y = "Assigned reads") +
    theme_minimal(base_size = 11) +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))
}


### Exploratory — PCA / scree / sample distance

In [ ]:
%%R -w 750 -h 550
pca_obj <- readRDS(rp("results", "exploratory", "pca.rds"))
scores <- pca_obj$scores
ve <- pca_obj$var_explained
ggplot(scores, aes(PC1, PC2, colour = condition, shape = as.factor(batch), label = sample)) +
  geom_point(size = 3) +
  geom_text(nudge_y = 0.04, size = 3, show.legend = FALSE) +
  labs(x = sprintf("PC1 (%.1f%%)", 100 * (ve[["PC1"]] %||% NA)),
       y = sprintf("PC2 (%.1f%%)", 100 * (ve[["PC2"]] %||% NA)),
       shape = "batch") +
  theme_minimal(base_size = 12)


In [ ]:
%%R -w 750 -h 400
ve <- pca_obj$var_explained
scree <- head(data.frame(PC = factor(names(ve), levels = names(ve)),
                          pct = as.numeric(ve) * 100),
              min(10, length(ve)))
ggplot(scree, aes(PC, pct)) + geom_col(fill = "#4c7fb0") +
  labs(x = NULL, y = "% variance explained") +
  theme_minimal(base_size = 12)


In [ ]:
%%R -w 750 -h 650
cor_obj <- readRDS(rp("results", "exploratory", "sample_correlation.rds"))
M <- cor_obj$dist
ord <- if (!is.null(cor_obj$hclust)) cor_obj$hclust$order else seq_len(nrow(M))
M <- M[ord, ord, drop = FALSE]
df <- as.data.frame(as.table(M))
names(df) <- c("x", "y", "distance")
df$x <- factor(df$x, levels = rownames(M)); df$y <- factor(df$y, levels = rownames(M))
ggplot(df, aes(x, y, fill = distance)) +
  geom_tile() +
  geom_text(aes(label = sprintf("%.1f", distance)), size = 3, colour = "grey20") +
  scale_fill_gradient(low = "#4c7fb0", high = "#ffffff",
                      limits = c(0, max(df$distance, na.rm = TRUE))) +
  labs(x = NULL, y = NULL, fill = "distance") +
  theme_minimal(base_size = 12) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))


### DE — volcano · MA · top-30 heatmap

아래 `padj_cutoff` / `lfc_cutoff` 값을 바꿔 DEG cutoff를 조절할 수 있음.

In [ ]:
#@markdown ### DE cutoff
padj_cutoff = 0.05  #@param {type:"number"}
lfc_cutoff = 1.0  #@param {type:"number"}

In [ ]:
%%R -i padj_cutoff -i lfc_cutoff
de_tbl <- safe_read(rp("results", "de", CONTRAST, "deseq2_results.csv"))
dds <- readRDS(rp("results", "exploratory", "dds_vst.rds"))
vst_mat <- tryCatch(SummarizedExperiment::assay(dds),
                    error = function(e) as.matrix(dds@assays@data[[1]]))

summary_path <- rp("results", "de", CONTRAST, "deg_summary.tsv")
safe_read(summary_path, sep = "\t")


In [ ]:
%%R -w 850 -h 600
v_df <- de_tbl |>
  dplyr::mutate(neglog10p = -log10(pmax(pvalue, 1e-300)),
                sig = !is.na(padj) & padj < padj_cutoff & abs(log2FoldChange) >= lfc_cutoff)
sig_df <- v_df |> dplyr::filter(sig)
ns_df  <- v_df |> dplyr::filter(!sig)
if (nrow(ns_df) > 3000) { set.seed(42); ns_df <- ns_df[sample.int(nrow(ns_df), 3000), ] }

ggplot() +
  geom_point(data = ns_df,  aes(log2FoldChange, neglog10p),
             colour = "grey70", alpha = 0.45, size = 1.1) +
  geom_point(data = sig_df, aes(log2FoldChange, neglog10p),
             colour = "#c0392b", alpha = 0.85, size = 1.6) +
  geom_vline(xintercept = c(-lfc_cutoff, lfc_cutoff), linetype = "dashed", colour = "grey40") +
  geom_hline(yintercept = -log10(padj_cutoff),       linetype = "dashed", colour = "grey40") +
  labs(title = CONTRAST, x = "log2FoldChange", y = "-log10(pvalue)",
       subtitle = sprintf("sig: padj<%.2g & |LFC|>=%.2g", padj_cutoff, lfc_cutoff)) +
  theme_minimal(base_size = 12)


In [ ]:
%%R -w 850 -h 900
top <- de_tbl |>
  dplyr::filter(!is.na(padj), !is.na(log2FoldChange)) |>
  dplyr::arrange(padj) |>
  head(30)

rows <- intersect(top$gene_id, rownames(vst_mat))
M <- vst_mat[rows, , drop = FALSE]
lbl <- setNames(top$gene_name, top$gene_id)
rn  <- ifelse(is.na(lbl[rows]) | !nzchar(lbl[rows]), rows, lbl[rows])
rownames(M) <- make.unique(rn)
Z <- t(scale(t(M)))
row_hc <- hclust(dist(Z),    method = "average")
col_hc <- hclust(dist(t(Z)), method = "average")
long <- as.data.frame(as.table(Z)); names(long) <- c("gene", "sample", "z")
long$gene   <- factor(long$gene,   levels = rownames(Z)[row_hc$order])
long$sample <- factor(long$sample, levels = colnames(Z)[col_hc$order])

ggplot(long, aes(sample, gene, fill = z)) +
  geom_tile() +
  scale_fill_gradient2(low = "#2c7bb6", mid = "white", high = "#d7191c", midpoint = 0) +
  labs(x = NULL, y = NULL, fill = "z-score") +
  theme_minimal(base_size = 11) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        axis.text.y = element_text(size = 9))


### GSEA — collection별 top 10 up/down

In [ ]:
%%R -w 900 -h 600
gsea <- safe_read(rp("results", "enrichment", CONTRAST, "gsea_combined.csv"))
if (is.null(gsea) || nrow(gsea) == 0) {
  no_results("GSEA returned no results.")
} else {
  .coll <- cfg$enrichment$gsea$collections %||% list()
  .lbl  <- setNames(vapply(.coll, function(x) x$label %||% x$id, character(1)),
                    vapply(.coll, function(x) x$id, character(1)))
  coll_lbl <- function(id) { out <- unname(.lbl[id]); ifelse(is.na(out), id, out) }
  for (c in unique(as.character(gsea$collection))) {
    df <- gsea |> dplyr::filter(collection == c) |>
      dplyr::mutate(direction = ifelse(NES > 0, "up", "down")) |>
      dplyr::group_by(direction) |>
      dplyr::arrange(padj, .by_group = TRUE) |>
      dplyr::slice_head(n = 10) |>
      dplyr::ungroup() |>
      dplyr::mutate(label = .trunc(pathway))
    if (nrow(df) == 0) next
    df$label <- factor(df$label, levels = rev(unique(df$label)))
    print(ggplot(df, aes(NES, label, fill = direction)) +
      geom_col() +
      scale_fill_manual(values = c(up = "#c0392b", down = "#2c7bb6")) +
      geom_vline(xintercept = 0, colour = "grey40") +
      labs(x = "NES", y = NULL,
           title = sprintf("%s — Top 10 up / Top 10 down", coll_lbl(c))) +
      theme_minimal(base_size = 11))
  }
}


### ORA — DB별 top 10 up/down

In [ ]:
%%R -w 900 -h 700
ora <- safe_read(rp("results", "enrichment", CONTRAST, "ora_combined.csv"))
if (is.null(ora) || nrow(ora) == 0) {
  no_results("ORA returned no results.")
} else {
  .db <- cfg$enrichment$ora$databases %||% list()
  .lbl <- setNames(vapply(.db, function(x) x$label %||% x$id, character(1)),
                   vapply(.db, function(x) x$id, character(1)))
  db_lbl <- function(id) { out <- unname(.lbl[id]); ifelse(is.na(out), id, out) }
  for (db in unique(ora$database)) {
    df <- ora |> dplyr::filter(database == db, direction %in% c("up", "down")) |>
      dplyr::group_by(direction) |>
      dplyr::arrange(p.adjust, .by_group = TRUE) |>
      dplyr::slice_head(n = 10) |>
      dplyr::ungroup() |>
      dplyr::mutate(
        neglog10p   = -log10(pmax(p.adjust, 1e-300)),
        Description = ifelse(is.na(Description) | !nzchar(Description), ID, Description),
        Description = .trunc(Description, 50))
    if (nrow(df) == 0) next
    print(ggplot(df, aes(neglog10p, reorder(Description, neglog10p), fill = direction)) +
      geom_col(position = position_dodge(width = 0.8)) +
      scale_fill_manual(values = c(up = "#c0392b", down = "#2c7bb6")) +
      labs(x = "-log10(p.adjust)", y = NULL,
           title = sprintf("%s — Top 10 up / Top 10 down", db_lbl(db))) +
      theme_minimal(base_size = 11))
  }
}


### TFEA — |score| 기준 상위 30 TF

In [ ]:
%%R -w 700 -h 800
tf <- safe_read(rp("results", "tfea", CONTRAST, "tf_scores.tsv"), sep = "\t")
if (is.null(tf) || nrow(tf) == 0) {
  no_results("TFEA results not available.")
} else {
  top <- tf |> dplyr::slice_max(abs(score), n = 30) |>
    dplyr::mutate(tf = forcats::fct_reorder(source, score))
  ggplot(top, aes(score, tf, fill = score > 0)) +
    geom_col() +
    scale_fill_manual(values = c(`TRUE` = "#c0392b", `FALSE` = "#2c7bb6"), guide = "none") +
    labs(title = paste("Top TFEA —", CONTRAST), y = NULL) +
    theme_minimal(base_size = 11)
}


### PROGENy — pathway 활성도 (contrast 단위, Wald stat 기반 MLM)

Pathway별 MLM activity score bar chart. score > 0이면 numerator에서 활성, < 0이면 denominator에서 활성.

In [ ]:
%%R -w 750 -h 600
pw <- safe_read(rp("results", "progeny", CONTRAST, "progeny_scores.tsv"), sep = "\t")
if (is.null(pw) || nrow(pw) == 0 ||
    !all(c("pathway", "score") %in% names(pw)) ||
    all(is.na(pw$score))) {
  no_results("PROGENy scores unavailable.")
} else {
  df <- pw |>
    dplyr::filter(!is.na(score)) |>
    dplyr::mutate(direction = ifelse(score > 0, "up", "down"),
                  pathway   = factor(pathway, levels = pathway[order(score)]))
  ggplot(df, aes(x = score, y = pathway, fill = direction)) +
    geom_col() +
    scale_fill_manual(values = c(up = "#c0392b", down = "#2c7bb6"), guide = "none") +
    geom_vline(xintercept = 0, colour = "grey40") +
    labs(x = "MLM activity score", y = NULL,
         title = "PROGENy pathway activity",
         subtitle = sprintf("%s — input: DESeq2 Wald stat", CONTRAST)) +
    theme_minimal(base_size = 12)
}


#### Score 표 (|score| 기준 정렬)

In [ ]:
%%R
if (is.null(pw) || nrow(pw) == 0 || !"score" %in% names(pw)) {
  no_results("PROGENy table unavailable.")
} else {
  pw |> dplyr::arrange(dplyr::desc(abs(score)))
}

### cMap — 상위 30 perturbagen

#### L2S2 input gene signatures

In [ ]:
%%R
top_up   <- cfg$cmap$top_up   %||% 250
top_down <- cfg$cmap$top_down %||% 250

if (is.null(de_tbl) || nrow(de_tbl) == 0) {
  no_results("DE results unavailable; L2S2 input signature cannot be reconstructed.")
} else {
  df <- de_tbl |>
    dplyr::filter(!is.na(gene_name), !is.na(log2FoldChange), !is.na(stat),
                  nzchar(as.character(gene_name)))
  up_tbl <- df |> dplyr::filter(log2FoldChange > 0) |>
    dplyr::slice_max(order_by = stat, n = top_up, with_ties = FALSE) |>
    dplyr::distinct(gene_name, .keep_all = TRUE) |>
    dplyr::mutate(direction = "up", input_rank = dplyr::row_number())
  down_tbl <- df |> dplyr::filter(log2FoldChange < 0) |>
    dplyr::slice_min(order_by = stat, n = top_down, with_ties = FALSE) |>
    dplyr::distinct(gene_name, .keep_all = TRUE) |>
    dplyr::mutate(direction = "down", input_rank = dplyr::row_number())
  sig_tbl <- dplyr::bind_rows(up_tbl, down_tbl) |>
    dplyr::select(dplyr::any_of(c("direction", "input_rank", "gene_id", "gene_name",
                                   "log2FoldChange", "stat", "pvalue", "padj")))
  cat(sprintf("L2S2 input: %d up / %d down genes (ranked by |Wald stat|)\n",
              sum(sig_tbl$direction == "up"), sum(sig_tbl$direction == "down")))
  sig_tbl
}


In [ ]:
%%R -w 850 -h 800
hits <- safe_read(rp("results", "cmap", CONTRAST, "l2s2_hits.tsv"), sep = "\t")
if (is.null(hits) || nrow(hits) == 0) {
  no_results("L2S2 query returned no hits.")
} else {
  top <- hits |>
    dplyr::arrange(dplyr::desc(abs(score %||% 0)), fdr) |>
    head(30) |>
    dplyr::mutate(label = ifelse(is.na(perturbagen) | !nzchar(perturbagen),
                                 paste0("row_", dplyr::row_number()), perturbagen),
                  label = factor(label, levels = rev(label)))
  ggplot(top, aes(score, label, fill = ifelse(score > 1, "induce", "reverse"))) +
    geom_col() +
    scale_fill_manual(values = c(induce = "#e67e22", reverse = "#2c7bb6"), name = "direction") +
    geom_vline(xintercept = 1, linetype = "dashed", colour = "grey40") +
    labs(x = "odds ratio (score)", y = NULL, title = "Top 30 perturbagens") +
    theme_minimal(base_size = 11)
}


### GSEA enrichment plot (interactive)

특정 MSigDB Gene set의 enrichment curve plot. fgsea/msigdbr는 섹션 11 setup 셀에서 이미 설치됨. 아래 pathway 이름을 바꿔서 다시 실행.

In [ ]:
%%R
suppressPackageStartupMessages({ library(fgsea); library(msigdbr) })

.ranking  <- cfg$enrichment$gsea$ranking %||% "stat"
.species  <- cfg$enrichment$species      %||% "Homo sapiens"
.gsea_combined <- safe_read(rp("results", "enrichment", CONTRAST, "gsea_combined.csv"))

.build_ranks <- function(de_res, method) {
  if (method == "stat") {
    rdf <- de_res |> dplyr::filter(!is.na(gene_name), !is.na(stat)) |>
      dplyr::group_by(gene_name) |> dplyr::summarise(metric = mean(stat), .groups = "drop")
  } else if (method == "signed_p") {
    rdf <- de_res |> dplyr::filter(!is.na(gene_name), !is.na(log2FoldChange), !is.na(pvalue)) |>
      dplyr::mutate(metric = sign(log2FoldChange) * -log10(pmax(pvalue, 1e-300))) |>
      dplyr::filter(is.finite(metric)) |>
      dplyr::group_by(gene_name) |> dplyr::summarise(metric = mean(metric), .groups = "drop")
  } else stop("Unsupported ranking: ", method)
  setNames(dplyr::arrange(rdf, dplyr::desc(metric))$metric,
           dplyr::arrange(rdf, dplyr::desc(metric))$gene_name)
}

.get_pathways <- function(coll_str, species) {
  parts   <- unlist(strsplit(coll_str, ":"))
  coll    <- parts[1]
  subcoll <- if (length(parts) > 1) paste(parts[-1], collapse = ":") else NULL
  m <- if (!is.null(subcoll)) msigdbr(species = species, collection = coll, subcollection = subcoll)
       else                   msigdbr(species = species, collection = coll)
  split(m$gene_symbol, m$gs_name)
}

.pathway_cache <- new.env(parent = emptyenv())

.detect_collection <- function(pathway) {
  prefixes <- c(HALLMARK_ = "H", REACTOME_ = "C2:CP:REACTOME",
                WP_ = "C2:CP:WIKIPATHWAYS", PID_ = "C2:CP:PID",
                BIOCARTA_ = "C2:CP:BIOCARTA")
  for (p in names(prefixes)) if (startsWith(pathway, p)) return(unname(prefixes[[p]]))
  if (!is.null(.gsea_combined)) {
    hit <- .gsea_combined[.gsea_combined$pathway == pathway, "collection", drop = TRUE]
    if (length(hit) > 0) return(as.character(hit[1]))
  }
  stop(sprintf("Could not infer collection for '%s'.", pathway))
}

# DE 셀보다 이 셀이 먼저 실행돼도 죽지 않도록 ranks 계산을 lazy 처리.
.ranks <- NULL
.get_ranks <- function() {
  if (is.null(.ranks)) {
    if (!exists("de_tbl") || is.null(de_tbl))
      stop("de_tbl not loaded — DE 섹션을 먼저 실행하세요.")
    .ranks <<- .build_ranks(de_tbl, .ranking)
  }
  .ranks
}

plot_gsea <- function(pathway, collection = NULL) {
  if (is.null(collection)) collection <- .detect_collection(pathway)
  if (is.null(.pathway_cache[[collection]]))
    .pathway_cache[[collection]] <- .get_pathways(collection, .species)
  sets <- .pathway_cache[[collection]]
  if (!pathway %in% names(sets))
    stop(sprintf("'%s' not found in collection %s.", pathway, collection))
  genes <- sets[[pathway]]
  ranks <- .get_ranks()
  row <- if (!is.null(.gsea_combined))
    .gsea_combined[.gsea_combined$collection == collection & .gsea_combined$pathway == pathway, ]
  else NULL
  nes_padj <- if (!is.null(row) && nrow(row) == 1)
    sprintf(" · NES=%.2f · padj=%.2g", row$NES, row$padj) else ""
  fgsea::plotEnrichment(genes, ranks) +
    ggplot2::labs(title = pathway,
                  subtitle = sprintf("%s · %s · ranking=%s · |set ∩ ranked|=%d%s",
                                     CONTRAST, collection, .ranking,
                                     sum(genes %in% names(ranks)), nes_padj))
}

list_pathways <- function(query = NULL, n = 20) {
  if (is.null(.gsea_combined) || nrow(.gsea_combined) == 0) return(.gsea_combined)
  all <- .gsea_combined |> dplyr::select(collection, pathway, NES, padj, size)
  if (!is.null(query)) all <- all[grepl(query, all$pathway, ignore.case = TRUE), ]
  dplyr::arrange(all, padj) |> head(n)
}

show_leading <- function(pathway, collection = NULL) {
  if (is.null(collection)) collection <- .detect_collection(pathway)
  if (is.null(.gsea_combined)) { message("GSEA combined CSV not found."); return(invisible()) }
  row <- .gsea_combined[.gsea_combined$collection == collection & .gsea_combined$pathway == pathway, ]
  if (nrow(row) == 0) { message("Pathway not in saved GSEA results."); return(invisible()) }
  le <- strsplit(as.character(row$leadingEdge[1]), ";", fixed = TRUE)[[1]]
  cat(sprintf("%s · %s\nNES=%.3f · padj=%.3g · size=%d · leadingEdge (%d):\n\n",
              pathway, collection, row$NES, row$padj, row$size, length(le)))
  cat(paste(le, collapse = ", "), "\n")
}
invisible(NULL)


아래 `plot_gsea(...)` 의 pathway 이름을 `gsea_combined.csv` 내 특정 gene set으로 바꾸면 plot을 그림.

In [ ]:
#@markdown ### GSEA pathway
gsea_pathway = "HALLMARK_INTERFERON_GAMMA_RESPONSE"  #@param {type:"string"}
#@markdown collection 자동 감지가 안 될 때만 수동 지정 (예: `C2:CGP`)
gsea_collection_override = ""  #@param {type:"string"}

In [ ]:
%%R -w 800 -h 500 -i gsea_pathway -i gsea_collection_override
if (nzchar(gsea_collection_override)) {
  plot_gsea(gsea_pathway, collection = gsea_collection_override)
} else {
  plot_gsea(gsea_pathway)
}


이 contrast에서 나온 pathway 탐색 — 키워드 필터, leading-edge 유전자 나열.

In [ ]:
%%R
list_pathways()                    # top 20 by padj across collections
# list_pathways("interferon")      # keyword filter
# list_pathways("apop", n = 50)
# show_leading("HALLMARK_INTERFERON_GAMMA_RESPONSE")


### L2S2 재쿼리 (signature 크기 조절)

top-up / top-down gene의 수를 조절해서 L2S2 api에 쿼리, 기본값은 각각 250개.

In [ ]:
%%R
## --- L2S2 re-query helpers (run once) ----------------------------------
suppressPackageStartupMessages({ library(httr); library(jsonlite) })

L2S2_ENDPOINT <- "https://l2s2.maayanlab.cloud/graphql"

.l2s2_paired_query <- '
query PairEnrichmentQuery($genesUp: [String]!, $genesDown: [String]!) {
  currentBackground {
    pairedEnrich(genesUp: $genesUp, genesDown: $genesDown) {
      consensus { drug oddsRatio pvalue adjPvalue }
    }
  }
}'

.l2s2_post <- function(payload) {
  resp <- httr::POST(L2S2_ENDPOINT, httr::content_type_json(),
                     body = jsonlite::toJSON(payload, auto_unbox = TRUE))
  httr::stop_for_status(resp)
  j <- httr::content(resp, as = "parsed", simplifyVector = FALSE)
  if (!is.null(j$errors))
    stop("GraphQL errors: ", jsonlite::toJSON(j$errors, auto_unbox = TRUE))
  j
}

# L2S2's pairedEnrich returns drug names only; MoA lives on FdaCount and the
# web UI joins them client-side. Mirror that with one aliased fdaCount batch.
.l2s2_fetch_moas <- function(drugs) {
  if (!length(drugs)) return(setNames(character(), character()))
  parts <- vapply(seq_along(drugs), function(i)
    sprintf('a%d: fdaCount(perturbation: %s) { moa }',
            i - 1L, jsonlite::toJSON(drugs[i], auto_unbox = TRUE)),
    character(1))
  query <- paste0("query MoaLookup { ", paste(parts, collapse = " "), " }")
  j <- tryCatch(.l2s2_post(list(query = query)),
                error = function(e) { message("MoA lookup failed: ", conditionMessage(e)); NULL })
  if (is.null(j)) return(setNames(rep("", length(drugs)), drugs))
  vals <- vapply(seq_along(drugs), function(i) {
    moa <- j$data[[paste0("a", i - 1L)]]$moa
    if (is.null(moa)) "" else as.character(moa)
  }, character(1))
  setNames(vals, drugs)
}

query_l2s2 <- function(top_up = 250, top_down = 250) {
  if (is.null(de_tbl)) stop("de_tbl not loaded — run the DE section first.")
  df <- de_tbl |>
    dplyr::filter(!is.na(gene_name), nzchar(as.character(gene_name)),
                  !is.na(log2FoldChange), !is.na(stat))
  up <- df |> dplyr::filter(log2FoldChange > 0) |>
    dplyr::slice_max(order_by = stat, n = top_up, with_ties = FALSE) |>
    dplyr::pull(gene_name) |> as.character() |> unique()
  down <- df |> dplyr::filter(log2FoldChange < 0) |>
    dplyr::slice_min(order_by = stat, n = top_down, with_ties = FALSE) |>
    dplyr::pull(gene_name) |> as.character() |> unique()
  if (!length(up) || !length(down))
    stop(sprintf("signature incomplete (up=%d down=%d); paired enrichment needs both.",
                 length(up), length(down)))
  cat(sprintf("Signature: %d up / %d down — querying %s\n",
              length(up), length(down), L2S2_ENDPOINT))

  payload <- list(query = .l2s2_paired_query,
                  variables = list(genesUp = I(up), genesDown = I(down)))
  j <- .l2s2_post(payload)
  cons <- j$data$currentBackground$pairedEnrich$consensus
  if (length(cons) == 0) { cat("No consensus hits.\n"); return(data.frame()) }

  out <- do.call(rbind, lapply(cons, function(r) data.frame(
    perturbagen = r$drug      %||% NA_character_,
    score       = r$oddsRatio %||% NA_real_,
    pvalue      = r$pvalue    %||% NA_real_,
    fdr         = r$adjPvalue %||% NA_real_,
    stringsAsFactors = FALSE)))
  out <- out[order(out$fdr, -out$score), , drop = FALSE]
  rownames(out) <- NULL

  drugs <- unique(out$perturbagen[!is.na(out$perturbagen) & nzchar(out$perturbagen)])
  moa_map <- .l2s2_fetch_moas(drugs)
  out$moa <- ifelse(out$perturbagen %in% names(moa_map),
                    moa_map[out$perturbagen], "")
  out$rank <- seq_len(nrow(out))
  out[, c("rank", "perturbagen", "moa", "score", "pvalue", "fdr")]
}

plot_l2s2 <- function(hits, n = 30, title = NULL) {
  if (is.null(hits) || nrow(hits) == 0) { no_results("No L2S2 hits."); return(invisible()) }
  top <- head(hits, n) |>
    dplyr::mutate(label = ifelse(is.na(perturbagen) | !nzchar(perturbagen),
                                 paste0("row_", dplyr::row_number()), perturbagen),
                  label = factor(label, levels = rev(label)))
  ggplot(top, aes(score, label, fill = ifelse(score > 1, "induce", "reverse"))) +
    geom_col() +
    scale_fill_manual(values = c(induce = "#e67e22", reverse = "#2c7bb6"), name = "direction") +
    geom_vline(xintercept = 1, linetype = "dashed", colour = "grey40") +
    labs(x = "odds ratio (score)", y = NULL,
         title = title %||% sprintf("Top %d perturbagens", nrow(top))) +
    theme_minimal(base_size = 11)
}

invisible(NULL)


In [ ]:
#@markdown ### L2S2 signature 크기
top_up = 100  #@param {type:"slider", min:50, max:500, step:50}
top_down = 100  #@param {type:"slider", min:50, max:500, step:50}
top_n_plot = 30  #@param {type:"slider", min:10, max:100, step:5}

In [ ]:
%%R -w 850 -h 800 -i top_up -i top_down -i top_n_plot
hits_custom <- query_l2s2(top_up = top_up, top_down = top_down)
plot_l2s2(hits_custom, n = top_n_plot,
          title = sprintf("Top %d perturbagens (up=%d, down=%d)", top_n_plot, top_up, top_down))
# head(hits_custom, 30)
